# 🎯 Training Setup - Loss Functions & Metrics

In [1]:
# Import required libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import time

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("✅ Libraries imported!")
print(f"🎯 Device: {device}")
if torch.cuda.is_available():
    print(f"💪 GPU: {torch.cuda.get_device_name(0)}")


✅ Libraries imported!
🎯 Device: cuda
💪 GPU: NVIDIA GeForce GTX 1650


## 🎲 Dice Loss Function

In [2]:
class DiceLoss(nn.Module):
    """
    Dice Loss for segmentation tasks
    Better for handling class imbalance in medical imaging
    """
    def __init__(self, smooth=1.0):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
    
    def forward(self, predictions, targets):
        # Flatten tensors
        predictions = predictions.reshape(-1)  # ✅ FIXED
        targets = targets.reshape(-1)          # ✅ FIXED
        
        # Calculate intersection and union
        intersection = (predictions * targets).sum()
        dice_score = (2. * intersection + self.smooth) / (predictions.sum() + targets.sum() + self.smooth)
        
        # Return dice loss (1 - dice score)
        return 1 - dice_score

print("✅ Dice Loss function defined!")
print("   Formula: 1 - (2 * intersection + smooth) / (pred + target + smooth)")


✅ Dice Loss function defined!
   Formula: 1 - (2 * intersection + smooth) / (pred + target + smooth)


## 🔀 Combined Loss Function

In [3]:
class CombinedLoss(nn.Module):
    """
    Combined Dice Loss + Binary Cross Entropy Loss
    Best for multi-class segmentation
    """
    def __init__(self, dice_weight=0.5, bce_weight=0.5):
        super(CombinedLoss, self).__init__()
        self.dice_loss = DiceLoss()
        self.bce_loss = nn.BCELoss()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
    
    def forward(self, predictions, targets):
        dice = self.dice_loss(predictions, targets)
        bce = self.bce_loss(predictions, targets)
        return self.dice_weight * dice + self.bce_weight * bce

# Initialize loss function
criterion = CombinedLoss(dice_weight=0.5, bce_weight=0.5)

print("✅ Combined Loss function created!")
print(f"   Dice weight: 0.5")
print(f"   BCE weight: 0.5")

✅ Combined Loss function created!
   Dice weight: 0.5
   BCE weight: 0.5


## 📊 Dice Score Metric

In [4]:

def dice_score(predictions, targets, threshold=0.5, smooth=1.0):
    """
    Calculate Dice Score (evaluation metric)
    Higher is better (0 to 1)
    """
    # Apply threshold
    predictions = (predictions > threshold).float()
    targets = targets.float()
    
    # Flatten - FIXED: view() → reshape()
    predictions = predictions.reshape(-1)  # ✅ FIXED
    targets = targets.reshape(-1)          # ✅ FIXED
    
    # Calculate intersection
    intersection = (predictions * targets).sum()
    
    # Calculate dice score
    dice = (2. * intersection + smooth) / (predictions.sum() + targets.sum() + smooth)
    
    return dice.item()

# Test dice score function
test_pred = torch.tensor([0.1, 0.9, 0.8, 0.2])
test_target = torch.tensor([0.0, 1.0, 1.0, 0.0])
test_dice = dice_score(test_pred, test_target)

print("✅ Dice Score function defined!")
print(f"   Test dice score: {test_dice:.4f}")

✅ Dice Score function defined!
   Test dice score: 1.0000


## ⚙️ Optimizer & Scheduler

In [5]:
# We'll need to load the model first
# Import model class from previous notebook
import sys
sys.path.append('..')

# For now, let's define optimizer parameters
learning_rate = 0.001
weight_decay = 1e-5

print("✅ Training hyperparameters set!")
print(f"\n📊 Hyperparameters:")
print(f"   Learning rate: {learning_rate}")
print(f"   Weight decay: {weight_decay}")
print(f"   Optimizer: Adam")
print(f"   Scheduler: ReduceLROnPlateau")
print(f"\n⚠️ Note: Model will be loaded from checkpoint or initialized in training notebook")


✅ Training hyperparameters set!

📊 Hyperparameters:
   Learning rate: 0.001
   Weight decay: 1e-05
   Optimizer: Adam
   Scheduler: ReduceLROnPlateau

⚠️ Note: Model will be loaded from checkpoint or initialized in training notebook


In [6]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """
    Train model for one epoch
    """
    model.train()
    epoch_loss = 0.0
    epoch_dice_wt = 0.0
    epoch_dice_tc = 0.0
    epoch_dice_et = 0.0
    
    progress_bar = tqdm(dataloader, desc="Training")
    
    for batch_idx, (images, masks) in enumerate(progress_bar):
        # Move to device
        images = images.to(device)
        masks = masks.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        
        # Calculate loss for each tumor region
        loss_wt = criterion(outputs[:, 0, :, :], masks[:, 0, :, :])
        loss_tc = criterion(outputs[:, 1, :, :], masks[:, 1, :, :])
        loss_et = criterion(outputs[:, 2, :, :], masks[:, 2, :, :])
        
        # Total loss
        loss = loss_wt + loss_tc + loss_et
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Calculate dice scores
        dice_wt = dice_score(outputs[:, 0, :, :], masks[:, 0, :, :])
        dice_tc = dice_score(outputs[:, 1, :, :], masks[:, 1, :, :])
        dice_et = dice_score(outputs[:, 2, :, :], masks[:, 2, :, :])
        
        # Accumulate metrics
        epoch_loss += loss.item()
        epoch_dice_wt += dice_wt
        epoch_dice_tc += dice_tc
        epoch_dice_et += dice_et
        
        # Update progress bar
        progress_bar.set_postfix({
            'Loss': f'{loss.item():.4f}',
            'Dice_WT': f'{dice_wt:.4f}'
        })
    
    # Calculate averages
    num_batches = len(dataloader)
    avg_loss = epoch_loss / num_batches
    avg_dice_wt = epoch_dice_wt / num_batches
    avg_dice_tc = epoch_dice_tc / num_batches
    avg_dice_et = epoch_dice_et / num_batches
    
    return avg_loss, avg_dice_wt, avg_dice_tc, avg_dice_et

print("✅ Training function defined!")


✅ Training function defined!


In [7]:
def validate(model, dataloader, criterion, device):
    """
    Validate model
    """
    model.eval()
    epoch_loss = 0.0
    epoch_dice_wt = 0.0
    epoch_dice_tc = 0.0
    epoch_dice_et = 0.0
    
    progress_bar = tqdm(dataloader, desc="Validation")
    
    with torch.no_grad():
        for images, masks in progress_bar:
            # Move to device
            images = images.to(device)
            masks = masks.to(device)
            
            # Forward pass
            outputs = model(images)
            
            # Calculate loss
            loss_wt = criterion(outputs[:, 0, :, :], masks[:, 0, :, :])
            loss_tc = criterion(outputs[:, 1, :, :], masks[:, 1, :, :])
            loss_et = criterion(outputs[:, 2, :, :], masks[:, 2, :, :])
            loss = loss_wt + loss_tc + loss_et
            
            # Calculate dice scores
            dice_wt = dice_score(outputs[:, 0, :, :], masks[:, 0, :, :])
            dice_tc = dice_score(outputs[:, 1, :, :], masks[:, 1, :, :])
            dice_et = dice_score(outputs[:, 2, :, :], masks[:, 2, :, :])
            
            # Accumulate metrics
            epoch_loss += loss.item()
            epoch_dice_wt += dice_wt
            epoch_dice_tc += dice_tc
            epoch_dice_et += dice_et
            
            # Update progress bar
            progress_bar.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'Dice_WT': f'{dice_wt:.4f}'
            })
    
    # Calculate averages
    num_batches = len(dataloader)
    avg_loss = epoch_loss / num_batches
    avg_dice_wt = epoch_dice_wt / num_batches
    avg_dice_tc = epoch_dice_tc / num_batches
    avg_dice_et = epoch_dice_et / num_batches
    
    return avg_loss, avg_dice_wt, avg_dice_tc, avg_dice_et

print("✅ Validation function defined!")


✅ Validation function defined!


In [8]:
def save_checkpoint(model, optimizer, epoch, loss, dice_scores, filepath):
    """
    Save model checkpoint
    """
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
        'dice_wt': dice_scores[0],
        'dice_tc': dice_scores[1],
        'dice_et': dice_scores[2]
    }
    torch.save(checkpoint, filepath)
    print(f"✅ Checkpoint saved: {filepath}")

def load_checkpoint(model, optimizer, filepath, device):
    """
    Load model checkpoint
    """
    checkpoint = torch.load(filepath, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']
    
    print(f"✅ Checkpoint loaded from epoch {epoch}")
    print(f"   Loss: {loss:.4f}")
    
    return model, optimizer, epoch

print("✅ Checkpoint functions defined!")
print("   save_checkpoint() - Save model")
print("   load_checkpoint() - Load model")


✅ Checkpoint functions defined!
   save_checkpoint() - Save model
   load_checkpoint() - Load model


## 🔧 Load Model, Data & Setup Training

In [9]:
# Import model and dataset classes
import os
import sys

# Re-define model classes (since we're in new notebook)
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=4, out_channels=3):
        super(UNet, self).__init__()
        self.encoder1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.encoder2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.encoder3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        self.encoder4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(512, 1024)
        self.upconv4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.decoder4 = DoubleConv(1024, 512)
        self.upconv3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.decoder3 = DoubleConv(512, 256)
        self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.decoder2 = DoubleConv(256, 128)
        self.upconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.decoder1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_channels, kernel_size=1)
    
    def forward(self, x):
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(self.pool1(enc1))
        enc3 = self.encoder3(self.pool2(enc2))
        enc4 = self.encoder4(self.pool3(enc3))
        bottleneck = self.bottleneck(self.pool4(enc4))
        dec4 = self.upconv4(bottleneck)
        dec4 = torch.cat([dec4, enc4], dim=1)
        dec4 = self.decoder4(dec4)
        dec3 = self.upconv3(dec4)
        dec3 = torch.cat([dec3, enc3], dim=1)
        dec3 = self.decoder3(dec3)
        dec2 = self.upconv2(dec3)
        dec2 = torch.cat([dec2, enc2], dim=1)
        dec2 = self.decoder2(dec2)
        dec1 = self.upconv1(dec2)
        dec1 = torch.cat([dec1, enc1], dim=1)
        dec1 = self.decoder1(dec1)
        out = self.out(dec1)
        return torch.sigmoid(out)

# Initialize model
model = UNet(in_channels=4, out_channels=3).to(device)

# Initialize optimizer
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# Initialize scheduler
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

print("✅ Model initialized and moved to GPU!")
print(f"✅ Optimizer: Adam (lr={learning_rate})")
print(f"✅ Scheduler: ReduceLROnPlateau")

✅ Model initialized and moved to GPU!
✅ Optimizer: Adam (lr=0.001)
✅ Scheduler: ReduceLROnPlateau


C:\Users\raj\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


## 📊 Load Data (Train & Validation)

In [10]:
 Load Data (Train & Validation)

✅ Data loaded successfully!
   Training samples: 64750
   Validation samples: 13912
   Batch size: 4
   Training batches: 16188
   Validation batches: 3478


## 🚀 Start Training!

In [11]:
# Training configuration
num_epochs = 5 # Start with 20 epochs
best_val_dice = 0.0
models_path = r'C:\Users\raj\Desktop\BrainTumorSegmentation\models'
os.makedirs(models_path, exist_ok=True)

# Training history
history = {
    'train_loss': [],
    'train_dice_wt': [],
    'train_dice_tc': [],
    'train_dice_et': [],
    'val_loss': [],
    'val_dice_wt': [],
    'val_dice_tc': [],
    'val_dice_et': []
}

print("🎯 Training Configuration:")
print(f"   Number of epochs: {num_epochs}")
print(f"   Batch size: {batch_size}")
print(f"   Learning rate: {learning_rate}")
print(f"   Device: {device}")
print(f"   Models save path: {models_path}")
print("\n" + "="*60)
print("🚀 STARTING TRAINING...")
print("="*60 + "\n")

# Start time
start_time = time.time()


🎯 Training Configuration:
   Number of epochs: 5
   Batch size: 4
   Learning rate: 0.001
   Device: cuda
   Models save path: C:\Users\raj\Desktop\BrainTumorSegmentation\models

🚀 STARTING TRAINING...



In [12]:
# Main training loop
for epoch in range(1, num_epochs + 1):
    print(f"\n{'='*60}")
    print(f"EPOCH {epoch}/{num_epochs}")
    print(f"{'='*60}")
    
    # Training phase
    train_loss, train_dice_wt, train_dice_tc, train_dice_et = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    
    # Validation phase
    val_loss, val_dice_wt, val_dice_tc, val_dice_et = validate(
        model, val_loader, criterion, device
    )
    
    # Update learning rate
    scheduler.step(val_loss)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_dice_wt'].append(train_dice_wt)
    history['train_dice_tc'].append(train_dice_tc)
    history['train_dice_et'].append(train_dice_et)
    history['val_loss'].append(val_loss)
    history['val_dice_wt'].append(val_dice_wt)
    history['val_dice_tc'].append(val_dice_tc)
    history['val_dice_et'].append(val_dice_et)
    
    # Print epoch results
    print(f"\n📊 Epoch {epoch} Results:")
    print(f"   Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"   Train Dice WT: {train_dice_wt:.4f} | Val Dice WT: {val_dice_wt:.4f}")
    print(f"   Train Dice TC: {train_dice_tc:.4f} | Val Dice TC: {val_dice_tc:.4f}")
    print(f"   Train Dice ET: {train_dice_et:.4f} | Val Dice ET: {val_dice_et:.4f}")
    
    # Save best model
    avg_val_dice = (val_dice_wt + val_dice_tc + val_dice_et) / 3
    if avg_val_dice > best_val_dice:
        best_val_dice = avg_val_dice
        best_model_path = os.path.join(models_path, 'best_model.pth')
        save_checkpoint(
            model, optimizer, epoch, val_loss, 
            (val_dice_wt, val_dice_tc, val_dice_et), 
            best_model_path
        )
        print(f"   🏆 New best model! Avg Dice: {avg_val_dice:.4f}")
    # Save checkpoint every 5 epochs
    if epoch % 5 == 0:
        checkpoint_path = os.path.join(models_path, f'checkpoint_epoch_{epoch}.pth')
        save_checkpoint(
            model, optimizer, epoch, val_loss,
            (val_dice_wt, val_dice_tc, val_dice_et),
            checkpoint_path
        )

# Training complete
end_time = time.time()
training_time = (end_time - start_time) / 60

print("\n" + "="*60)
print("🎉 TRAINING COMPLETE! 🎉")
print("="*60)
print(f"⏱️ Total training time: {training_time:.2f} minutes")
print(f"🏆 Best validation Dice score: {best_val_dice:.4f}")


EPOCH 1/5


Validation: 100%|███████████████████████████████████| 3478/3478 [1:35:25<00:00,  1.65s/it, Loss=0.2219, Dice_WT=1.0000]



📊 Epoch 1 Results:
   Train Loss: 0.5549 | Val Loss: 0.5089
   Train Dice WT: 0.7556 | Val Dice WT: 0.6887
   Train Dice TC: 0.6129 | Val Dice TC: 0.7377
   Train Dice ET: 0.6481 | Val Dice ET: 0.7643
✅ Checkpoint saved: C:\Users\raj\Desktop\BrainTumorSegmentation\models\best_model.pth
   🏆 New best model! Avg Dice: 0.7302

EPOCH 2/5


Validation: 100%|███████████████████████████████████| 3478/3478 [1:40:51<00:00,  1.74s/it, Loss=0.3332, Dice_WT=1.0000]



📊 Epoch 2 Results:
   Train Loss: 0.4048 | Val Loss: 0.4398
   Train Dice WT: 0.8117 | Val Dice WT: 0.7383
   Train Dice TC: 0.7187 | Val Dice TC: 0.8076
   Train Dice ET: 0.7473 | Val Dice ET: 0.8123
✅ Checkpoint saved: C:\Users\raj\Desktop\BrainTumorSegmentation\models\best_model.pth
   🏆 New best model! Avg Dice: 0.7861

EPOCH 3/5


...


📊 Epoch 3 Results:
   Train Loss: 0.3535 | Val Loss: 0.4347
   Train Dice WT: 0.8299 | Val Dice WT: 0.7467
   Train Dice TC: 0.7641 | Val Dice TC: 0.8067
   Train Dice ET: 0.7759 | Val Dice ET: 0.8175
✅ Checkpoint saved: C:\Users\raj\Desktop\BrainTumorSegmentation\models\best_model.pth
   🏆 New best model! Avg Dice: 0.7903

EPOCH 4/5


Validation: 100%|███████████████████████████████████| 3478/3478 [1:35:56<00:00,  1.66s/it, Loss=0.2033, Dice_WT=1.0000]



📊 Epoch 4 Results:
   Train Loss: 0.3230 | Val Loss: 0.4415
   Train Dice WT: 0.8437 | Val Dice WT: 0.7341
   Train Dice TC: 0.7887 | Val Dice TC: 0.8105
   Train Dice ET: 0.7915 | Val Dice ET: 0.8329
✅ Checkpoint saved: C:\Users\raj\Desktop\BrainTumorSegmentation\models\best_model.pth
   🏆 New best model! Avg Dice: 0.7925

EPOCH 5/5


Validation: 100%|███████████████████████████████████| 3478/3478 [1:36:12<00:00,  1.66s/it, Loss=0.1113, Dice_WT=1.0000]



📊 Epoch 5 Results:
   Train Loss: 0.3042 | Val Loss: 0.3932
   Train Dice WT: 0.8478 | Val Dice WT: 0.7589
   Train Dice TC: 0.8049 | Val Dice TC: 0.8491
   Train Dice ET: 0.8046 | Val Dice ET: 0.8526
✅ Checkpoint saved: C:\Users\raj\Desktop\BrainTumorSegmentation\models\best_model.pth
   🏆 New best model! Avg Dice: 0.8202
✅ Checkpoint saved: C:\Users\raj\Desktop\BrainTumorSegmentation\models\checkpoint_epoch_5.pth

🎉 TRAINING COMPLETE! 🎉
⏱️ Total training time: 3219.58 minutes
🏆 Best validation Dice score: 0.8202


In [3]:
# Import required libraries
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
import nibabel as nib
from torch.utils.data import Dataset, DataLoader
import random

# Set seeds
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("✅ Libraries imported!")
print(f"🎯 Device: {device}")
if torch.cuda.is_available():
    print(f"💪 GPU: {torch.cuda.get_device_name(0)}")

✅ Libraries imported!
🎯 Device: cuda
💪 GPU: NVIDIA GeForce GTX 1650


## 🧠 Re-define Model Classes

In [5]:
# Re-define model classes
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=4, out_channels=3):
        super(UNet, self).__init__()
        self.encoder1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.encoder2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.encoder3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        self.encoder4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(512, 1024)
        self.upconv4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.decoder4 = DoubleConv(1024, 512)
        self.upconv3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.decoder3 = DoubleConv(512, 256)
        self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.decoder2 = DoubleConv(256, 128)
        self.upconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.decoder1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, out_channels, kernel_size=1)
    
    def forward(self, x):
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(self.pool1(enc1))
        enc3 = self.encoder3(self.pool2(enc2))
        enc4 = self.encoder4(self.pool3(enc3))
        bottleneck = self.bottleneck(self.pool4(enc4))
        dec4 = self.upconv4(bottleneck)
        dec4 = torch.cat([dec4, enc4], dim=1)
        dec4 = self.decoder4(dec4)
        dec3 = self.upconv3(dec4)
        dec3 = torch.cat([dec3, enc3], dim=1)
        dec3 = self.decoder3(dec3)
        dec2 = self.upconv2(dec3)
        dec2 = torch.cat([dec2, enc2], dim=1)
        dec2 = self.decoder2(dec2)
        dec1 = self.upconv1(dec2)
        dec1 = torch.cat([dec1, enc1], dim=1)
        dec1 = self.decoder1(dec1)
        out = self.out(dec1)
        return torch.sigmoid(out)

# Initialize model
model = UNet(in_channels=4, out_channels=3).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

print("✅ Model and optimizer initialized!")

✅ Model and optimizer initialized!


## 📊 Re-define Loss Functions & Metrics

In [6]:
# Dice Loss
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
    
    def forward(self, predictions, targets):
        predictions = predictions.reshape(-1)
        targets = targets.reshape(-1)
        intersection = (predictions * targets).sum()
        dice_score = (2. * intersection + self.smooth) / (predictions.sum() + targets.sum() + self.smooth)
        return 1 - dice_score

# Combined Loss
class CombinedLoss(nn.Module):
    def __init__(self, dice_weight=0.5, bce_weight=0.5):
        super(CombinedLoss, self).__init__()
        self.dice_loss = DiceLoss()
        self.bce_loss = nn.BCELoss()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
    
    def forward(self, predictions, targets):
        dice = self.dice_loss(predictions, targets)
        bce = self.bce_loss(predictions, targets)
        return self.dice_weight * dice + self.bce_weight * bce

# Dice Score Metric
def dice_score(predictions, targets, threshold=0.5, smooth=1.0):
    predictions = (predictions > threshold).float()
    targets = targets.float()
    predictions = predictions.reshape(-1)
    targets = targets.reshape(-1)
    intersection = (predictions * targets).sum()
    dice = (2. * intersection + smooth) / (predictions.sum() + targets.sum() + smooth)
    return dice.item()

# Initialize loss
criterion = CombinedLoss(dice_weight=0.5, bce_weight=0.5)

print("✅ Loss functions and metrics defined!")


✅ Loss functions and metrics defined!


## 🔄 Training & Validation Functions

In [7]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    epoch_loss = 0.0
    epoch_dice_wt = 0.0
    epoch_dice_tc = 0.0
    epoch_dice_et = 0.0
    
    progress_bar = tqdm(dataloader, desc="Training")
    
    for batch_idx, (images, masks) in enumerate(progress_bar):
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        
        loss_wt = criterion(outputs[:, 0, :, :], masks[:, 0, :, :])
        loss_tc = criterion(outputs[:, 1, :, :], masks[:, 1, :, :])
        loss_et = criterion(outputs[:, 2, :, :], masks[:, 2, :, :])
        loss = loss_wt + loss_tc + loss_et
        
        loss.backward()
        optimizer.step()
        
        dice_wt = dice_score(outputs[:, 0, :, :], masks[:, 0, :, :])
        dice_tc = dice_score(outputs[:, 1, :, :], masks[:, 1, :, :])
        dice_et = dice_score(outputs[:, 2, :, :], masks[:, 2, :, :])
        
        epoch_loss += loss.item()
        epoch_dice_wt += dice_wt
        epoch_dice_tc += dice_tc
        epoch_dice_et += dice_et
        
        progress_bar.set_postfix({'Loss': f'{loss.item():.4f}', 'Dice_WT': f'{dice_wt:.4f}'})
    
    num_batches = len(dataloader)
    return epoch_loss/num_batches, epoch_dice_wt/num_batches, epoch_dice_tc/num_batches, epoch_dice_et/num_batches

def validate(model, dataloader, criterion, device):
    model.eval()
    epoch_loss = 0.0
    epoch_dice_wt = 0.0
    epoch_dice_tc = 0.0
    epoch_dice_et = 0.0
    
    progress_bar = tqdm(dataloader, desc="Validation")
    
    with torch.no_grad():
        for images, masks in progress_bar:
            images = images.to(device)
            masks = masks.to(device)
            outputs = model(images)
            
            loss_wt = criterion(outputs[:, 0, :, :], masks[:, 0, :, :])
            loss_tc = criterion(outputs[:, 1, :, :], masks[:, 1, :, :])
            loss_et = criterion(outputs[:, 2, :, :], masks[:, 2, :, :])
            loss = loss_wt + loss_tc + loss_et
            
            dice_wt = dice_score(outputs[:, 0, :, :], masks[:, 0, :, :])
            dice_tc = dice_score(outputs[:, 1, :, :], masks[:, 1, :, :])
            dice_et = dice_score(outputs[:, 2, :, :], masks[:, 2, :, :])
            
            epoch_loss += loss.item()
            epoch_dice_wt += dice_wt
            epoch_dice_tc += dice_tc
            epoch_dice_et += dice_et
            
            progress_bar.set_postfix({'Loss': f'{loss.item():.4f}', 'Dice_WT': f'{dice_wt:.4f}'})
    
    num_batches = len(dataloader)
    return epoch_loss/num_batches, epoch_dice_wt/num_batches, epoch_dice_tc/num_batches, epoch_dice_et/num_batches

def save_checkpoint(model, optimizer, epoch, loss, dice_scores, filepath):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
        'dice_wt': dice_scores[0],
        'dice_tc': dice_scores[1],
        'dice_et': dice_scores[2]
    }
    torch.save(checkpoint, filepath)
    print(f"✅ Checkpoint saved: {filepath}")

print("✅ Training functions defined!")

✅ Training functions defined!


## 📦 Load Data (Train & Validation)

In [8]:
# Normalization function
def normalize_scan(scan):
    brain_mask = scan > 0
    if np.sum(brain_mask) == 0:
        return scan
    brain_values = scan[brain_mask]
    min_val = np.min(brain_values)
    max_val = np.max(brain_values)
    if max_val > min_val:
        scan_norm = np.zeros_like(scan, dtype=np.float32)
        scan_norm[brain_mask] = (scan[brain_mask] - min_val) / (max_val - min_val)
        return scan_norm
    else:
        return scan.astype(np.float32)

# Dataset class
class BraTSDataset(Dataset):
    def __init__(self, patient_list, data_path, slice_range=(56, 130)):
        self.patient_list = patient_list
        self.data_path = data_path
        self.slice_range = slice_range
        self.samples = []
        for patient in patient_list:
            for slice_idx in range(slice_range[0], slice_range[1]):
                self.samples.append((patient, slice_idx))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        patient_id, slice_idx = self.samples[idx]
        patient_path = os.path.join(self.data_path, patient_id)
        
        flair = nib.load(os.path.join(patient_path, f'{patient_id}_flair.nii.gz')).get_fdata()[:, :, slice_idx]
        t1 = nib.load(os.path.join(patient_path, f'{patient_id}_t1.nii.gz')).get_fdata()[:, :, slice_idx]
        t1ce = nib.load(os.path.join(patient_path, f'{patient_id}_t1ce.nii.gz')).get_fdata()[:, :, slice_idx]
        t2 = nib.load(os.path.join(patient_path, f'{patient_id}_t2.nii.gz')).get_fdata()[:, :, slice_idx]
        seg = nib.load(os.path.join(patient_path, f'{patient_id}_seg.nii.gz')).get_fdata()[:, :, slice_idx]
        
        flair = normalize_scan(flair)
        t1 = normalize_scan(t1)
        t1ce = normalize_scan(t1ce)
        t2 = normalize_scan(t2)
        
        image = np.stack([flair, t1, t1ce, t2], axis=0)
        
        wt_mask = (seg > 0).astype(np.float32)
        tc_mask = ((seg == 1) | (seg == 4)).astype(np.float32)
        et_mask = (seg == 4).astype(np.float32)
        mask = np.stack([wt_mask, tc_mask, et_mask], axis=0)
        
        image = torch.from_numpy(image).float()
        mask = torch.from_numpy(mask).float()
        
        return image, mask

# Load patient splits
raw_path = r'C:\Users\raj\Desktop\BrainTumorSegmentation\data\raw'
processed_path = r'C:\Users\raj\Desktop\BrainTumorSegmentation\data\processed'

train_df = pd.read_csv(os.path.join(processed_path, 'train_patients.csv'))
val_df = pd.read_csv(os.path.join(processed_path, 'val_patients.csv'))

train_patients = train_df['patient_id'].tolist()
val_patients = val_df['patient_id'].tolist()

# Create datasets
train_dataset = BraTSDataset(train_patients, raw_path)
val_dataset = BraTSDataset(val_patients, raw_path)

# Create dataloaders
batch_size = 4
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print("✅ Data loaded successfully!")
print(f"   Training samples: {len(train_dataset)}")
print(f"   Validation samples: {len(val_dataset)}")
print(f"   Batch size: {batch_size}")

✅ Data loaded successfully!
   Training samples: 64750
   Validation samples: 13912
   Batch size: 4


## 🔄 Load Checkpoint & Resume Training


In [9]:
# Load checkpoint from epoch 5
checkpoint_path = r'C:\Users\raj\Desktop\BrainTumorSegmentation\models\checkpoint_epoch_5.pth'

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
start_epoch = checkpoint['epoch']
previous_loss = checkpoint['loss']

print("✅ Checkpoint Loaded Successfully!")
print(f"\n📊 Resuming from Epoch {start_epoch}")
print(f"   Previous Val Loss: {previous_loss:.4f}")
print(f"   Previous Dice Scores:")
print(f"   WT: {checkpoint['dice_wt']:.4f}")
print(f"   TC: {checkpoint['dice_tc']:.4f}")
print(f"   ET: {checkpoint['dice_et']:.4f}")
avg_dice = (checkpoint['dice_wt'] + checkpoint['dice_tc'] + checkpoint['dice_et'])/3
print(f"   Average: {avg_dice:.4f}")
print(f"\n🎯 Continuing training for 3 more epochs (6-8)...")


C:\Users\raj\AppData\Local\Temp\ipykernel_11204\1220470141.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=device)


✅ Checkpoint Loaded Successfully!

📊 Resuming from Epoch 5
   Previous Val Loss: 0.3932
   Previous Dice Scores:
   WT: 0.7589
   TC: 0.8491
   ET: 0.8526
   Average: 0.8202

🎯 Continuing training for 3 more epochs (6-8)...


## 🚀 Resume Training - Epochs 6, 7, 8

In [11]:
# Training configuration
num_epochs = 8  # Total epochs (5 done + 3 more)
best_val_dice = 0.8202  # Current best
models_path = r'C:\Users\raj\Desktop\BrainTumorSegmentation\models'
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

# Create history from previous 5 epochs manually
history = {
    'train_loss': [0.5549, 0.4048, 0.3535, 0.3230, 0.3042],
    'train_dice_wt': [0.7556, 0.8117, 0.8299, 0.8437, 0.8478],
    'train_dice_tc': [0.6129, 0.7187, 0.7641, 0.7887, 0.8049],
    'train_dice_et': [0.6481, 0.7473, 0.7759, 0.7915, 0.8046],
    'val_loss': [0.5089, 0.4398, 0.4347, 0.4415, 0.3932],
    'val_dice_wt': [0.6887, 0.7383, 0.7467, 0.7341, 0.7589],
    'val_dice_tc': [0.7377, 0.8076, 0.8067, 0.8105, 0.8491],
    'val_dice_et': [0.7643, 0.8123, 0.8175, 0.8329, 0.8526]
}

print("🎯 Training Configuration:")
print(f"   Resume from epoch: {start_epoch}")
print(f"   Target epochs: {num_epochs} (3 more)")
print(f"   Best validation Dice: {best_val_dice:.4f}")
print(f"   Device: {device}")
print(f"   Previous 5 epochs history loaded!")
print("\n" + "="*60)
print("🚀 RESUMING TRAINING...")
print("="*60 + "\n")

# Start time
start_time = time.time()

# Training loop for epochs 6, 7, 8
for epoch in range(start_epoch + 1, num_epochs + 1):
    print(f"\n{'='*60}")
    print(f"EPOCH {epoch}/{num_epochs}")
    print(f"{'='*60}")
    
    # Training phase
    train_loss, train_dice_wt, train_dice_tc, train_dice_et = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    
    # Validation phase
    val_loss, val_dice_wt, val_dice_tc, val_dice_et = validate(
        model, val_loader, criterion, device
    )
    
    # Update learning rate
    scheduler.step(val_loss)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_dice_wt'].append(train_dice_wt)
    history['train_dice_tc'].append(train_dice_tc)
    history['train_dice_et'].append(train_dice_et)
    history['val_loss'].append(val_loss)
    history['val_dice_wt'].append(val_dice_wt)
    history['val_dice_tc'].append(val_dice_tc)
    history['val_dice_et'].append(val_dice_et)
    
    # Print epoch results
    print(f"\n📊 Epoch {epoch} Results:")
    print(f"   Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"   Train Dice WT: {train_dice_wt:.4f} | Val Dice WT: {val_dice_wt:.4f}")
    print(f"   Train Dice TC: {train_dice_tc:.4f} | Val Dice TC: {val_dice_tc:.4f}")
    print(f"   Train Dice ET: {train_dice_et:.4f} | Val Dice ET: {val_dice_et:.4f}")
    
    # Save best model
    avg_val_dice = (val_dice_wt + val_dice_tc + val_dice_et) / 3
    if avg_val_dice > best_val_dice:
        best_val_dice = avg_val_dice
        best_model_path = os.path.join(models_path, 'best_model.pth')
        save_checkpoint(
            model, optimizer, epoch, val_loss, 
            (val_dice_wt, val_dice_tc, val_dice_et), 
            best_model_path
        )
        print(f"   🏆 New best model! Avg Dice: {avg_val_dice:.4f}")
    
    # Save checkpoint
    checkpoint_path = os.path.join(models_path, f'checkpoint_epoch_{epoch}.pth')
    save_checkpoint(
        model, optimizer, epoch, val_loss,
        (val_dice_wt, val_dice_tc, val_dice_et),
        checkpoint_path
    )

# Training complete
end_time = time.time()
training_time = (end_time - start_time) / 60

print("\n" + "="*60)
print("🎉 ADDITIONAL TRAINING COMPLETE! 🎉")
print("="*60)
print(f"⏱️ Additional training time: {training_time:.2f} minutes")
print(f"🏆 Best validation Dice score: {best_val_dice:.4f}")

# Save updated history
history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(models_path, 'training_history.csv'), index=False)
print(f"✅ Training history saved!")

🎯 Training Configuration:
   Resume from epoch: 5
   Target epochs: 8 (3 more)
   Best validation Dice: 0.8202
   Device: cuda
   Previous 5 epochs history loaded!

🚀 RESUMING TRAINING...


EPOCH 6/8


Validation: 100%|███████████████████████████████████| 3478/3478 [1:39:48<00:00,  1.72s/it, Loss=0.2927, Dice_WT=1.0000]



📊 Epoch 6 Results:
   Train Loss: 0.2865 | Val Loss: 0.5035
   Train Dice WT: 0.8569 | Val Dice WT: 0.6733
   Train Dice TC: 0.8185 | Val Dice TC: 0.7792
   Train Dice ET: 0.8132 | Val Dice ET: 0.7769
✅ Checkpoint saved: C:\Users\raj\Desktop\BrainTumorSegmentation\models\checkpoint_epoch_6.pth

EPOCH 7/8


Training:   2%|▌                                    | 253/16188 [09:18<9:46:36,  2.21s/it, Loss=0.1707, Dice_WT=0.8721]


KeyboardInterrupt: 